# Làm sạch dữ liệu bằng PySpark

Chạy từ trên xuống bằng kernel container audit. Quy tắc chi tiết nằm trong docs/data_cleaning.md.

Xóa dòng thiếu children, ADR âm, ADR trên 1000 và bản sao dư của dòng trùng hoàn toàn. Thống nhất country: CN thành CHN, TMP thành TLS. Điền Unknown cho country, agent, company. Giữ dòng có tổng khách bằng 0, tổng đêm bằng 0 hoặc ADR bằng 0. CSV kết quả gồm 32 cột nguồn và cột arrival_date; báo cáo JSON lưu kết quả kiểm tra.


Hai thời gian được đo khi chạy **Run All** liên tục:

- **Toàn quy trình:** khởi động Spark, đọc và chuẩn hóa đầu vào, làm sạch, kiểm tra rồi ghi CSV lên HDFS. Báo cáo tách riêng thời gian khởi động.
- **Chỉ xử lý:** làm sạch và kiểm tra sau khi dữ liệu đã được đọc/chuẩn hóa, không tính khởi động Spark hay đọc/ghi HDFS. Dữ liệu đầu vào và kết quả trung gian được cache để Spark không phải đọc lại HDFS cho từng action.

Thời gian xử lý kéo dài qua nhiều cell; không dừng lâu giữa các cell. Mặc định notebook kết nối Spark Standalone gồm ba worker. Chạy `./scripts/start_notebooks.sh local` nếu muốn chạy local[*]. Khi đổi chế độ, Restart Kernel trước khi chạy lại notebook.


## 1. Cấu hình

In [1]:
import json
import os
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F

INPUT_FILE = os.environ.get(
    "AUDIT_HDFS_FILE",
    "/data/hotel_bookings/input/hotel_bookings_synthetic_200MB.csv",
)
INPUT_PATH = f"hdfs://namenode:9000{INPUT_FILE}"
SPARK_MASTER = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_PATH = f"hdfs://namenode:9000/data/hotel_bookings/cleaned/pyspark/{RUN_ID}"

NUMERIC_COLUMNS = [
    "is_canceled", "lead_time", "arrival_date_year",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "days_in_waiting_list", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
]
CATEGORY_COLUMNS = ["country", "agent", "company"]
COUNTRY_CODE_MAP = {"CN": "CHN", "TMP": "TLS"}
INTEGER_COLUMNS = [column for column in NUMERIC_COLUMNS if column != "adr"]
BINARY_COLUMNS = ["is_canceled", "is_repeated_guest"]
VALID_RANGES = {
    "arrival_date_year": (1, 9999),
    "arrival_date_week_number": (1, 53),
    "arrival_date_day_of_month": (1, 31),
}
MONTHS = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}
CANONICAL_VALUES = {
    "hotel": ["City Hotel", "Resort Hotel"],
    "reservation_status": ["Canceled", "Check-Out", "No-Show"],
    "arrival_date_month": list(MONTHS),
}
SOURCE_COLUMNS = [
    "hotel", "is_canceled", "lead_time", "arrival_date_year", "arrival_date_month",
    "arrival_date_week_number", "arrival_date_day_of_month", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children", "babies", "meal", "country",
    "market_segment", "distribution_channel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled", "reserved_room_type",
    "assigned_room_type", "booking_changes", "deposit_type", "agent", "company",
    "days_in_waiting_list", "customer_type", "adr", "required_car_parking_spaces",
    "total_of_special_requests", "reservation_status", "reservation_status_date",
]
MAX_SAFE_INTEGER = 2**53 - 1
RULES_VERSION = "hotel-clean-v5"
BENCHMARK_VERSION = "clean-benchmark-v1"
MISSING_WORDS = ["", "nan", "null", "none", "na", "n/a"]
REPORT_DIR = Path("/workspace/reports")


## 2. Đọc dữ liệu và chuẩn hóa null

Đọc đúng 32 cột gốc. Chuẩn hóa khoảng trắng, chuỗi thiếu và mã quốc gia (CN → CHN, TMP → TLS). Sau đó so sánh trùng trên toàn bộ 32 cột; giữ một dòng, xóa các bản sao còn lại.


In [2]:
total_started_at = time.perf_counter()
spark = (
    SparkSession.builder
    .appName("HotelBookingsCleaning")
    .master(SPARK_MASTER)
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "audit"))
    .config("spark.driver.bindAddress", os.environ.get("SPARK_DRIVER_BIND_ADDRESS", "0.0.0.0"))
    .config("spark.executor.cores", os.environ.get("SPARK_EXECUTOR_CORES", "1"))
    .config("spark.cores.max", os.environ.get("SPARK_CORES_MAX", "3"))
    .config("spark.ui.port", "4040")
    .config("spark.sql.ansi.enabled", "true")
    .getOrCreate()
)
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.sparkContext.setLogLevel("ERROR")
spark_master = spark.sparkContext.master
spark_mode = "distributed" if spark_master.startswith("spark://") else "local"
print(f"Chế độ Spark: {spark_mode} ({spark_master})")
spark_startup_seconds = time.perf_counter() - total_started_at
pipeline_started_at = time.perf_counter()

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .option("escape", '"')
    .csv(INPUT_PATH)
)
column_names = [column.strip() for column in raw.columns]
if len(set(column_names)) != len(column_names) or set(column_names) != set(SOURCE_COLUMNS):
    raise ValueError("CSV phải có đúng 32 cột gốc, không chứa cột trùng hoặc cột kết quả.")
raw = raw.toDF(*column_names)

normalized = raw.select(*[
    F.when(
        F.col(column).isNull()
        | F.lower(F.regexp_replace(F.col(column), r"(?U)^\s+|\s+$", "")).isin(*MISSING_WORDS),
        F.lit(None),
    ).otherwise(F.regexp_replace(F.col(column), r"(?U)^\s+|\s+$", "")).alias(column)
    for column in SOURCE_COLUMNS
]).persist(StorageLevel.MEMORY_AND_DISK)
input_rows = int(normalized.count())
source_read_seconds = time.perf_counter() - pipeline_started_at
processing_started_at = time.perf_counter()

cleaned = normalized.withColumn("country", F.upper(F.col("country")))
cleaned = cleaned.replace(COUNTRY_CODE_MAP, subset=["country"])

duplicate_window = Window.partitionBy(*SOURCE_COLUMNS).orderBy(F.lit(1))
cleaned = cleaned.withColumn("__duplicate_rank", F.row_number().over(duplicate_window))
before = cleaned.agg(
    F.sum(F.when(F.col("__duplicate_rank") > 1, 1).otherwise(0)).alias("duplicate_rows_removed"),
    *[
        F.sum(F.when(
            (F.col("__duplicate_rank") == 1) & F.col(column).isNull(), 1
        ).otherwise(0)).alias(column)
        for column in CATEGORY_COLUMNS
    ],
).first().asDict()
before["input_rows"] = input_rows
duplicate_rows_removed = int(before["duplicate_rows_removed"] or 0)
filled_missing = {column: int(before[column] or 0) for column in CATEGORY_COLUMNS}
cleaned = cleaned.where(F.col("__duplicate_rank") == 1).drop("__duplicate_rank")
cleaned = cleaned.fillna({column: "Unknown" for column in CATEGORY_COLUMNS})

print(f"Số dòng đầu vào: {input_rows:,}")
print(f"Bản sao trùng đã xóa: {duplicate_rows_removed:,}")
display(pd.DataFrame(filled_missing.items(), columns=["column", "filled_with_Unknown"]))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/22 09:31:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Chế độ Spark: distributed (spark://spark-master:7077)


[Stage 7:============================================>              (3 + 1) / 4]

Số dòng đầu vào: 1,383,291
Bản sao trùng đã xóa: 53,675


,column,filled_with_Unknown
0,country,5411
1,agent,179880
2,company,1254703


## 3. Làm sạch số và xóa dòng theo quy tắc

Giá trị số không hợp lệ được chuyển thành null. Sau đó xóa dòng thiếu `children`, có ADR âm hoặc ADR trên 1000. ADR không đọc được được giữ ở dạng null.

In [3]:
def any_true(conditions):
    result = F.lit(False)
    for condition in conditions:
        result = result | F.coalesce(condition, F.lit(False))
    return result

numeric_values = {}
for column in NUMERIC_COLUMNS:
    parsed = F.expr(f"try_cast({column} as double)")
    invalid = F.col(column).isNotNull() & (
        parsed.isNull() | F.isnan(parsed) | (F.abs(parsed) == float("inf"))
    )
    values = F.when(invalid, F.lit(None)).otherwise(parsed)

    if column == "adr":
        numeric_values[column] = values
        continue

    invalid_count = (values < 0) | ((values % 1) != 0) | (values > MAX_SAFE_INTEGER)
    invalid_binary = F.lit(False)
    invalid_range = F.lit(False)
    if column in BINARY_COLUMNS:
        invalid_binary = values.isNotNull() & ~values.isin(0, 1)
    if column in VALID_RANGES:
        lower, upper = VALID_RANGES[column]
        invalid_range = (values < lower) | (values > upper)

    numeric_values[column] = F.when(
        any_true([invalid_count, invalid_binary, invalid_range]), F.lit(None)
    ).otherwise(values).cast("long")

cleaned = cleaned.select(*[
    numeric_values.get(column, F.col(column)).alias(column)
    for column in SOURCE_COLUMNS
])
removal_stats = cleaned.agg(
    F.sum(F.when(F.col("children").isNull(), 1).otherwise(0)).alias("children_missing"),
    F.sum(F.when(
        F.col("children").isNotNull() & (F.col("adr") < 0), 1
    ).otherwise(0)).alias("negative_adr"),
    F.sum(F.when(
        F.col("children").isNotNull() & (F.col("adr") > 1000), 1
    ).otherwise(0)).alias("adr_above_1000"),
).first().asDict()
removed_children_missing = int(removal_stats["children_missing"] or 0)
removed_negative_adr = int(removal_stats["negative_adr"] or 0)
removed_adr_above_1000 = int(removal_stats["adr_above_1000"] or 0)
cleaned = cleaned.where(
    F.col("children").isNotNull()
    & (F.col("adr").isNull() | ((F.col("adr") >= 0) & (F.col("adr") <= 1000)))
)

## 4. Chuẩn hóa mã phân loại

Chuẩn hóa hotel, trạng thái, tháng và country. Mã `agent`/`company` dạng `9.00` được chuẩn hóa thành `9`; các giá trị khác được giữ nguyên.

In [4]:
canonical_columns = {}
for column, allowed_values in CANONICAL_VALUES.items():
    lookup = F.create_map(*[
        item for value in allowed_values
        for item in (F.lit(value.lower()), F.lit(value))
    ])
    canonical = lookup[F.lower(F.col(column))]
    canonical_columns[column] = F.coalesce(canonical, F.col(column))
cleaned = cleaned.select(*[
    canonical_columns.get(column, F.col(column)).alias(column)
    for column in cleaned.columns
])

cleaned = cleaned.withColumn(
    "country",
    F.when(F.lower(F.col("country")) == "unknown", "Unknown").otherwise(F.upper(F.col("country"))),
)
for column in ["agent", "company"]:
    values = F.when(F.lower(F.col(column)) == "unknown", "Unknown").otherwise(F.col(column))
    cleaned = cleaned.withColumn(
        column, F.regexp_replace(values, r"^([0-9]+)\.0+$", "$1")
    )

## 5. Chuẩn hóa ngày

Tạo `arrival_date` theo định dạng ISO. Ngày không hợp lệ được chuyển thành null; các giá trị ngày còn lại được chuẩn hóa.

In [5]:
month_map = F.create_map(*[
    item for month, number in MONTHS.items()
    for item in (F.lit(month), F.lit(number))
])
arrival_text = F.concat(
    F.lpad(F.col("arrival_date_year").cast("string"), 4, "0"), F.lit("-"),
    F.lpad(month_map[F.col("arrival_date_month")].cast("string"), 2, "0"), F.lit("-"),
    F.lpad(F.col("arrival_date_day_of_month").cast("string"), 2, "0"),
)
cleaned = cleaned.withColumn("__arrival_text", arrival_text)
cleaned = cleaned.withColumn("__arrival_date", F.expr("try_cast(__arrival_text as date)"))
cleaned = cleaned.withColumn(
    "__status_date",
    F.when(
        F.col("reservation_status_date").rlike(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$"),
        F.expr("try_cast(reservation_status_date as date)"),
    ),
)
cleaned = cleaned.withColumn(
    "arrival_date", F.date_format(F.col("__arrival_date"), "yyyy-MM-dd")
).withColumn(
    "reservation_status_date", F.date_format(F.col("__status_date"), "yyyy-MM-dd")
).drop("__arrival_text", "__arrival_date", "__status_date")

## 6. Kiểm tra và tổng hợp

Tổng khách bằng 0, tổng đêm bằng 0 và ADR bằng 0 được giữ nguyên. ADR trên 1000 đã bị xóa ở bước trước.

In [6]:
cleaned = cleaned.persist(StorageLevel.MEMORY_AND_DISK)

## 7. Kiểm tra kết quả trước khi ghi

Notebook kiểm tra các quy tắc xóa, kiểu số, giá trị phân loại và số null còn lại. Nếu kiểm tra thất bại, notebook dừng trước khi ghi dữ liệu.

In [7]:
def count_when(condition, name):
    return F.sum(F.when(condition, 1).otherwise(0)).alias(name)

check_conditions = {
    "nonfinite_numeric": any_true([
        F.isnan(F.col(column).cast("double")) | (F.abs(F.col(column)) == float("inf"))
        for column in NUMERIC_COLUMNS
    ]),
    "invalid_counts": any_true([
        (F.col(column) < 0) | ((F.col(column) % 1) != 0)
        for column in INTEGER_COLUMNS
    ]),
    "invalid_binary": any_true([
        F.col(column).isNotNull() & ~F.col(column).isin(0, 1)
        for column in BINARY_COLUMNS
    ]),
    "invalid_range": any_true([
        (F.col(column) < lower) | (F.col(column) > upper)
        for column, (lower, upper) in VALID_RANGES.items()
    ]),
    "negative_adr": F.col("adr") < 0,
    "adr_above_1000": F.col("adr") > 1000,
    "null_categories": any_true([F.col(column).isNull() for column in CATEGORY_COLUMNS]),
    "country_aliases_remaining": F.col("country").isin(*COUNTRY_CODE_MAP),
}
after = cleaned.agg(
    F.count(F.lit(1)).alias("output_rows"),
    *[
        count_when(F.col(column).isNull(), f"missing__{column}")
        for column in SOURCE_COLUMNS + ["arrival_date"]
    ],
    *[count_when(condition, f"check__{name}") for name, condition in check_conditions.items()],
).first().asDict()

remaining_missing = {
    column: int(after[f"missing__{column}"] or 0)
    for column in SOURCE_COLUMNS + ["arrival_date"]
}
post_clean_checks = {
    "row_count_difference": abs(
        (int(before["input_rows"]) - duplicate_rows_removed
         - removed_children_missing - removed_negative_adr
         - removed_adr_above_1000)
        - int(after["output_rows"])
    ),
    "children_missing": int(after["missing__children"] or 0),
    **{name: int(after[f"check__{name}"] or 0) for name in check_conditions},
}
processing_seconds = time.perf_counter() - processing_started_at
display(pd.DataFrame(post_clean_checks.items(), columns=["check", "remaining_errors"]))
if any(post_clean_checks.values()):
    raise ValueError("Kết quả chưa đạt kiểm tra; dừng trước khi ghi HDFS.")

display(pd.DataFrame(remaining_missing.items(), columns=["column", "remaining_nulls"]))

,check,remaining_errors
0,row_count_difference,0
1,children_missing,0
2,nonfinite_numeric,0
3,invalid_counts,0
4,invalid_binary,0
5,invalid_range,0
6,negative_adr,0
7,adr_above_1000,0
8,null_categories,0
9,country_aliases_remaining,0


,column,remaining_nulls
0,hotel,0
1,is_canceled,0
2,lead_time,0
3,arrival_date_year,0
4,arrival_date_month,0
5,arrival_date_week_number,0
6,arrival_date_day_of_month,0
7,stays_in_weekend_nights,0
8,stays_in_week_nights,0
9,adults,0


## 8. Ghi CSV mới lên HDFS

Spark tạo thư mục chứa các part-*.csv và _SUCCESS. Đọc lại bằng cách truyền cả đường dẫn thư mục. Không gộp thành một file để giữ cách ghi dữ liệu của Spark.

In [8]:
if any(post_clean_checks.values()):
    raise ValueError("Kết quả chưa đạt kiểm tra; dừng trước khi ghi HDFS.")

write_started_at = time.perf_counter()
(
    cleaned.select(*SOURCE_COLUMNS, "arrival_date").write
    .mode("errorifexists")
    .option("header", True)
    .option("nullValue", "")
    .option("escape", '"')
    .csv(OUTPUT_PATH)
)
hdfs_write_seconds = time.perf_counter() - write_started_at
pipeline_seconds = time.perf_counter() - pipeline_started_at
total_seconds = time.perf_counter() - total_started_at
cleaned.unpersist(blocking=True)
normalized.unpersist(blocking=True)

summary = {
    "engine": "pyspark",
    "benchmark_version": BENCHMARK_VERSION,
    "spark_mode": spark_mode,
    "spark_master": spark_master,
    "source": INPUT_FILE,
    "output": OUTPUT_PATH,
    "input_rows": int(before["input_rows"]),
    "output_rows": int(after["output_rows"]),
    "filled_missing": filled_missing,
    "rules_version": RULES_VERSION,
    "post_clean_checks": post_clean_checks,
    "remaining_missing": remaining_missing,
    "removed_rows": {
        "exact_duplicate_extra": duplicate_rows_removed,
        "children_missing": removed_children_missing,
        "negative_adr": removed_negative_adr,
        "adr_above_1000": removed_adr_above_1000,
        "total": duplicate_rows_removed + removed_children_missing + removed_negative_adr + removed_adr_above_1000,
    },
    "duplicate_policy": "drop_extra_keep_first",
    "source_read_seconds": round(source_read_seconds, 3),
    "processing_seconds": round(processing_seconds, 3),
    "hdfs_write_seconds": round(hdfs_write_seconds, 3),
    "read_clean_write_seconds": round(pipeline_seconds, 3),
    "pipeline_seconds": round(pipeline_seconds, 3),
    "spark_startup_seconds": round(spark_startup_seconds, 3),
    "total_seconds": round(total_seconds, 3),
}
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "clean_pyspark.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Đã ghi thư mục: {OUTPUT_PATH}")
print(f"Đọc và chuẩn hóa: {source_read_seconds:.2f} giây")
print(f"Làm sạch và kiểm tra: {processing_seconds:.2f} giây")
print(f"Ghi HDFS: {hdfs_write_seconds:.2f} giây")
print(f"Khởi động Spark: {spark_startup_seconds:.2f} giây")
print(f"Toàn quy trình, chưa tính khởi động: {pipeline_seconds:.2f} giây")
print(f"Tổng thời gian gồm khởi động Spark: {total_seconds:.2f} giây")


Đã ghi thư mục: hdfs://namenode:9000/data/hotel_bookings/cleaned/pyspark/20261022_093121_135423
Đọc và chuẩn hóa: 26.31 giây
Làm sạch và kiểm tra: 37.96 giây
Ghi HDFS: 9.03 giây
Khởi động Spark: 5.06 giây
Toàn quy trình, chưa tính khởi động: 73.34 giây
Tổng thời gian gồm khởi động Spark: 78.40 giây


## 9. Hai phép so sánh với Pandas

Chạy lại `clean_pandas.ipynb` trước, sau đó chạy PySpark bằng **Run All**. Bảng đầu tiên so sánh toàn bộ quy trình (đọc, làm sạch, kiểm tra, ghi HDFS và tách riêng thời gian khởi động Spark). Bảng thứ hai chỉ so sánh làm sạch và kiểm tra sau khi dữ liệu đã được chuẩn bị trong bộ nhớ. Cả hai phép đo đều yêu cầu chạy liên tục, không dừng lâu giữa các cell.


In [9]:
pandas_summary_file = REPORT_DIR / "clean_pandas.json"
if not pandas_summary_file.exists():
    print("Chưa có reports/clean_pandas.json. Chạy clean_pandas.ipynb trước.")
else:
    pandas_summary = json.loads(pandas_summary_file.read_text(encoding="utf-8"))
    if pandas_summary.get("benchmark_version") != BENCHMARK_VERSION:
        print("Báo cáo Pandas đang dùng định dạng thời gian cũ. Chạy lại clean_pandas.ipynb rồi mới đối chiếu.")
    else:
        timing_rows = [
            {
                "engine": "Pandas",
                "chế độ thực thi": "local",
                "đọc + chuẩn hóa (s)": pandas_summary["source_read_seconds"],
                "làm sạch + kiểm tra (s)": pandas_summary["processing_seconds"],
                "ghi HDFS (s)": pandas_summary["hdfs_write_seconds"],
                "khởi động Spark (s)": pandas_summary["spark_startup_seconds"],
                "toàn pipeline (s)": pandas_summary["pipeline_seconds"],
                "end-to-end gồm startup (s)": pandas_summary["total_seconds"],
            },
            {
                "engine": "PySpark",
                "chế độ thực thi": f"{summary['spark_mode']} ({summary['spark_master']})",
                "đọc + chuẩn hóa (s)": summary["source_read_seconds"],
                "làm sạch + kiểm tra (s)": summary["processing_seconds"],
                "ghi HDFS (s)": summary["hdfs_write_seconds"],
                "khởi động Spark (s)": summary["spark_startup_seconds"],
                "toàn pipeline (s)": summary["pipeline_seconds"],
                "end-to-end gồm startup (s)": summary["total_seconds"],
            },
        ]
        print("So sánh toàn quy trình: gồm đọc, làm sạch, kiểm tra và ghi HDFS.")
        display(pd.DataFrame(timing_rows))
        print("So sánh riêng xử lý: không tính đọc/chuẩn hóa, ghi HDFS hay khởi động Spark.")
        display(pd.DataFrame([
            {"engine": row["engine"], "làm sạch + kiểm tra (s)": row["làm sạch + kiểm tra (s)"]}
            for row in timing_rows
        ]))

    display(pd.DataFrame([
        {
            "engine": item["engine"],
            "input_rows": item["input_rows"],
            "output_rows": item["output_rows"],
            "rules_version": item.get("rules_version"),
        }
        for item in [pandas_summary, summary]
    ]))
    display(pd.DataFrame([
        {
            "rule": rule,
            "pandas_removed": pandas_summary.get("removed_rows", {}).get(rule),
            "pyspark_removed": summary.get("removed_rows", {}).get(rule),
        }
        for rule in ["exact_duplicate_extra", "children_missing", "negative_adr", "adr_above_1000", "total"]
    ]))
    if pandas_summary["source"] != summary["source"]:
        print("Hai báo cáo đang đọc hai đường dẫn đầu vào khác nhau.")
    if pandas_summary.get("rules_version") != RULES_VERSION:
        print("Hai báo cáo đang dùng quy tắc làm sạch khác phiên bản; chưa thể đối chiếu trực tiếp.")
    display(pd.DataFrame([
        {
            "column": column,
            "pandas_nulls": pandas_summary.get("remaining_missing", {}).get(column),
            "pyspark_nulls": remaining_missing[column],
        }
        for column in SOURCE_COLUMNS + ["arrival_date"]
    ]))
    print("Số liệu tóm tắt khớp chưa chứng minh từng dòng trong hai CSV giống hệt nhau.")


So sánh toàn quy trình: gồm đọc, làm sạch, kiểm tra và ghi HDFS.


,engine,chế độ thực thi,đọc + chuẩn hóa (s),làm sạch + kiểm tra (s),ghi HDFS (s),khởi động Spark (s),toàn pipeline (s),end-to-end gồm startup (s)
0,Pandas,local,11.965,17.142,7.507,0.000,36.625,36.625
1,PySpark,distributed (spark://spark-master:7077),26.314,37.955,9.033,5.059,73.339,78.399


So sánh riêng xử lý: không tính đọc/chuẩn hóa, ghi HDFS hay khởi động Spark.


,engine,làm sạch + kiểm tra (s)
0,Pandas,17.142
1,PySpark,37.955


,engine,input_rows,output_rows,rules_version
0,pandas,1383291,1329545,hotel-clean-v5
1,pyspark,1383291,1329545,hotel-clean-v5


,rule,pandas_removed,pyspark_removed
0,exact_duplicate_extra,53675,53675
1,children_missing,54,54
2,negative_adr,12,12
3,adr_above_1000,5,5
4,total,53746,53746


,column,pandas_nulls,pyspark_nulls
0,hotel,0,0
1,is_canceled,0,0
2,lead_time,0,0
3,arrival_date_year,0,0
4,arrival_date_month,0,0
5,arrival_date_week_number,0,0
6,arrival_date_day_of_month,0,0
7,stays_in_weekend_nights,0,0
8,stays_in_week_nights,0,0
9,adults,0,0


Số liệu tóm tắt khớp chưa chứng minh từng dòng trong hai CSV giống hệt nhau.
